### ouverture des fichiers et jointures 

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from fonction import *

In [2]:
fp = Path("data") / "raw" / "communes" / "ADE_4-0_GPKG_WGS84G_FRA-ED2025-11-20.gpkg"
if not fp.exists():
	raise FileNotFoundError(f"File not found: {fp.resolve()}")
communes = gpd.read_file(fp, layer='commune')

fp = Path("data") / "intermediate" / "datagouv" / "swimonthly_combined_1958to2025.gpkg"
if not fp.exists():
	raise FileNotFoundError(f"File not found: {fp.resolve()}")
swi_monthly = gpd.read_file(fp)


fp = Path("data") / "intermediate" / "meteo_france" / "meteo_france_combined.gpkg"
if not fp.exists():
    raise FileNotFoundError(f"File not found: {fp.resolve()}")
meteo_france = gpd.read_file(fp)


fp = Path("data") / "intermediate" / "argile" / "argile.gpkg"
if not fp.exists():
    raise FileNotFoundError(f"File not found: {fp.resolve()}")
argile = gpd.read_file(fp)

In [3]:
display(communes.head())

,cleabs,nom_officiel,nom_officiel_en_majuscules,statut,code_insee,population,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region,codes_siren_des_epci,code_siren,code_postal,superficie_cadastrale,geometry
0,COMMUNE_0000000000001001,L'Abergement-Clémenciat,L'ABERGEMENT-CLEMENCIAT,Commune simple,01001,859,2022-01-01,INSEE,0108,012,01,84,200069193,210100012,01400,1590,"MULTIPOLYGON (((4.95841 46.15327, 4.95812 46.1..."
1,COMMUNE_0000000000001002,L'Abergement-de-Varey,L'ABERGEMENT-DE-VAREY,Commune simple,01002,273,2022-01-01,INSEE,0101,011,01,84,240100883,210100020,01640,920,"MULTIPOLYGON (((5.4302 45.98277, 5.43012 45.98..."
2,COMMUNE_0000000000001004,Ambérieu-en-Bugey,AMBERIEU-EN-BUGEY,Commune simple,01004,15554,2022-01-01,INSEE,0101,011,01,84,240100883,210100046,01500,2460,"MULTIPOLYGON (((5.40882 45.94206, 5.4085 45.94..."
3,COMMUNE_0000000000001005,Ambérieux-en-Dombes,AMBERIEUX-EN-DOMBES,Commune simple,01005,1917,2022-01-01,INSEE,0122,012,01,84,200042497,210100053,01330,1590,"MULTIPOLYGON (((4.94298 45.97962, 4.94257 45.9..."
4,COMMUNE_0000000000001006,Ambléon,AMBLEON,Commune simple,01006,114,2022-01-01,INSEE,0104,011,01,84,200040350,210100061,01300,590,"MULTIPOLYGON (((5.57083 45.75338, 5.57219 45.7..."


In [4]:
swi_monthly['DATE'] = pd.to_datetime(swi_monthly['DATE']) - pd.offsets.MonthBegin(1)
swi_monthly = swi_monthly[swi_monthly['DATE'] >= '1960-01-01']
display(swi_monthly)

,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry
168164,1960-01-01,600,24010,5.5,98.9,7.396774,5.145161,5.581742,87057.7,8987.6,...,0.000355,0.0,0.035,0.006452,5.3,0.319419,0.000065,-2.1,13.3,POINT (60000 2401000)
168165,1960-01-01,760,23610,0.6,91.8,7.477419,6.616129,5.709613,86828.0,9751.3,...,0.000000,0.0,0.001,0.000000,0.6,0.263484,0.000194,-1.5,12.6,POINT (76000 2361000)
168166,1960-01-01,760,23930,0.3,91.5,7.583871,6.558065,5.743839,87214.9,9749.5,...,0.000000,0.0,0.001,0.000000,0.4,0.318710,0.000129,-1.4,12.9,POINT (76000 2393000)
168167,1960-01-01,760,24010,5.9,101.1,7.064516,5.238710,5.480290,85955.7,8996.8,...,0.000355,0.0,0.030,0.012903,5.7,0.321290,0.000323,-2.2,12.6,POINT (76000 2401000)
168168,1960-01-01,760,24090,5.8,100.3,7.196774,5.203226,5.517258,86361.2,8993.0,...,0.000290,0.0,0.027,0.012903,5.6,0.321452,0.000323,-2.1,12.8,POINT (76000 2409000)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7982859,2025-10-01,11960,17050,0.0,44.5,18.122581,2.538710,8.597774,91740.2,33351.6,...,0.000000,0.0,0.000,0.000000,0.0,0.190774,0.000000,12.5,25.8,POINT (1196000 1705000)
7982860,2025-10-01,11960,17130,0.0,27.8,16.987097,2.683871,7.945290,92372.0,33203.6,...,0.000000,0.0,0.000,0.000000,0.0,0.194065,0.000000,9.3,25.8,POINT (1196000 1713000)
7982861,2025-10-01,11960,17210,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.000000,0.0,0.000,0.000000,0.0,0.186806,0.000000,9.3,25.8,POINT (1196000 1721000)
7982862,2025-10-01,11960,17290,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.000000,0.0,0.000,0.000000,0.0,0.204161,0.000000,9.3,25.8,POINT (1196000 1729000)


In [5]:
display(meteo_france)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,geometry
0,2,641374,7106309,1960-01-01,"0,863",POINT (641374 7106309)
1,2,641374,7106309,1960-02-01,"0,876",POINT (641374 7106309)
2,2,641374,7106309,1960-03-01,"0,856",POINT (641374 7106309)
3,2,641374,7106309,1960-04-01,"0,757",POINT (641374 7106309)
4,2,641374,7106309,1960-05-01,"0,673",POINT (641374 7106309)
...,...,...,...,...,...,...
7005175,9892,1215772,6046242,2024-08-01,"-0,019",POINT (1215772 6046242)
7005176,9892,1215772,6046242,2024-09-01,"0,007",POINT (1215772 6046242)
7005177,9892,1215772,6046242,2024-10-01,"0,17",POINT (1215772 6046242)
7005178,9892,1215772,6046242,2024-11-01,"0,126",POINT (1215772 6046242)


In [6]:
print(argile.crs)
cols_argile = ["geometry", "ALEA", "NIVEAU", "DPT"]
argile=argile.to_crs(swi_monthly.crs)

EPSG:3857


In [7]:
print(meteo_france.crs)
cols_meteo = ["DATE", "SWI_UNIF_MENS"]
meteo_france=meteo_france.to_crs(swi_monthly.crs)
print(swi_monthly.crs)
print(meteo_france.crs)

EPSG:2154
EPSG:27572
EPSG:27572


In [24]:
swi_monthly

,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry
0,1958-08-01,600,24010,0.0,101.1,16.183871,3.735484,10.297871,104077.3,31484.2,...,0.0,0.0,0.0,0.0,0.0,0.234903,0.0,10.3,21.4,POINT (60000 2401000)
1,1958-08-01,760,23610,0.0,85.0,16.277419,4.748387,10.441677,103165.5,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.174645,0.0,11.5,21.5,POINT (76000 2361000)
2,1958-08-01,760,23930,0.0,84.1,16.400000,4.719355,10.511032,103705.2,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.226355,0.0,11.6,21.8,POINT (76000 2393000)
3,1958-08-01,760,24010,0.0,103.3,15.819355,3.800000,10.094516,102541.1,31493.2,...,0.0,0.0,0.0,0.0,0.0,0.220677,0.0,10.2,20.6,POINT (76000 2401000)
4,1958-08-01,760,24090,0.0,102.4,15.958065,3.764516,10.168677,103106.1,31489.5,...,0.0,0.0,0.0,0.0,0.0,0.217355,0.0,10.2,20.9,POINT (76000 2409000)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7982859,2025-10-01,11960,17050,0.0,44.5,18.122581,2.538710,8.597774,91740.2,33351.6,...,0.0,0.0,0.0,0.0,0.0,0.190774,0.0,12.5,25.8,POINT (1196000 1705000)
7982860,2025-10-01,11960,17130,0.0,27.8,16.987097,2.683871,7.945290,92372.0,33203.6,...,0.0,0.0,0.0,0.0,0.0,0.194065,0.0,9.3,25.8,POINT (1196000 1713000)
7982861,2025-10-01,11960,17210,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.0,0.0,0.0,0.0,0.0,0.186806,0.0,9.3,25.8,POINT (1196000 1721000)
7982862,2025-10-01,11960,17290,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.0,0.0,0.0,0.0,0.0,0.204161,0.0,9.3,25.8,POINT (1196000 1729000)


In [23]:
meteo_france

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,geometry
0,2,641374,7106309,1960-01-01,"0,863",POINT (588001.454 2673000.704)
1,2,641374,7106309,1960-02-01,"0,876",POINT (588001.454 2673000.704)
2,2,641374,7106309,1960-03-01,"0,856",POINT (588001.454 2673000.704)
3,2,641374,7106309,1960-04-01,"0,757",POINT (588001.454 2673000.704)
4,2,641374,7106309,1960-05-01,"0,673",POINT (588001.454 2673000.704)
...,...,...,...,...,...,...
7005175,9892,1215772,6046242,2024-08-01,"-0,019",POINT (1171999.296 1617001.255)
7005176,9892,1215772,6046242,2024-09-01,"0,007",POINT (1171999.296 1617001.255)
7005177,9892,1215772,6046242,2024-10-01,"0,17",POINT (1171999.296 1617001.255)
7005178,9892,1215772,6046242,2024-11-01,"0,126",POINT (1171999.296 1617001.255)


### jointure des bases


### Explication de la jointure

La jointure effectuée dans la cellule ci-dessous combine les données météorologiques de `meteo_france` avec les données de l'indice SWI (Soil Wetness Index) de `swi_monthly`. Voici les étapes principales de cette jointure :

1. **Groupement par date** :
    - Les données de `meteo_france` sont regroupées par la colonne `DATE`. Cela permet de traiter les données pour chaque mois séparément.

2. **Filtrage des données SWI** :
    - Pour chaque date, les données correspondantes dans `swi_monthly` sont filtrées pour ne conserver que celles ayant la même date que le groupe courant de `meteo_france`.

3. **Jointure spatiale** :
    - Une jointure spatiale est réalisée entre les données météorologiques (`meteo_d`) et les données SWI filtrées (`swi_d`) en utilisant la méthode `sjoin_nearest` de GeoPandas.
    - Cette méthode associe chaque point de `meteo_d` au point le plus proche dans `swi_d`, en fonction de leur distance géographique.
    - Une distance maximale (`max_distance=100`) est définie pour limiter l'association à des points proches (à ajuster selon la résolution des données).

4. **Concaténation des résultats** :
    - Les résultats de chaque jointure (par date) sont concaténés pour former un GeoDataFrame final, `gdf_join`.

5. **Nettoyage des colonnes** :
    - Les colonnes inutiles issues de la jointure, comme `LAMBX_right`, `LAMBY_right`, `geometry_right`, et `index_right`, sont supprimées.
    - Les colonnes restantes sont renommées pour éviter les doublons et clarifier les données.

6. **Export des données** :
    - Le GeoDataFrame final est sauvegardé dans un fichier GeoPackage (`jointure_meteo_swimonthly_full.gpkg`) pour une utilisation ultérieure.

Cette jointure permet de combiner les informations météorologiques et les indices SWI pour chaque point géographique, facilitant ainsi l'analyse des relations entre les conditions météorologiques et l'humidité des sols.
```

In [8]:
output_fp = Path("data") / "processed" / "jointure_meteo_swimonthly_full.gpkg"
output_fp.parent.mkdir(parents=True, exist_ok=True)



if output_fp.exists():
    print(" Jointure déjà existante — chargement depuis le disque.")
    gdf_join = gpd.read_file(output_fp)



else:
    print(" Jointure inexistante — calcul de la jointure spatio-temporelle.")

    res = []

    for d, meteo_d in meteo_france.groupby("DATE"):
        swi_d = swi_monthly[swi_monthly["DATE"] == d]
        if swi_d.empty:
            continue

        tmp = gpd.sjoin_nearest(
            meteo_d,
            swi_d.drop(columns=["DATE"]),   # évite DATE_left / DATE_right
            how="left",
            max_distance=100,               # à adapter à la maille
        )

        res.append(tmp)

    gdf_join = gpd.GeoDataFrame(
        pd.concat(res, ignore_index=True),
        crs=meteo_france.crs
    )


    gdf_join = (
        gdf_join
        .sort_values("DATE")
        .drop_duplicates(subset=["NUMERO", "DATE"], keep="first")
    )


    cols_to_drop = [
        "LAMBX_right",
        "LAMBY_right",
        "geometry_right",
        "index_right",
    ]

    gdf_join = gdf_join.drop(
        columns=[c for c in cols_to_drop if c in gdf_join.columns]
    )

    gdf_join = gdf_join.rename(columns={
        "LAMBX_left": "LAMBX",
        "LAMBY_left": "LAMBY"
    })


    gdf_join.to_file(output_fp, driver="GPKG")
    print(f" Jointure sauvegardée : {output_fp}")



display(gdf_join)

 Jointure inexistante — calcul de la jointure spatio-temporelle.
 Jointure sauvegardée : data\processed\jointure_meteo_swimonthly_full.gpkg


,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,geometry,PRENEI,PRELIQ,T,FF,...,RESR_NEIGE6,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H
0,2,641374,7106309,1960-01-01,"0,863",POINT (588001.454 2673000.704),4.9,51.5,4.829032,6.822581,...,0.303226,0.001710,0.001839,0.024,0.096774,4.0,0.287032,0.002484,-7.2,11.6
5997,7119,635809,6442801,1960-01-01,"1,081",POINT (587999.304 2008998.388),20.2,96.4,1.990323,2.061290,...,4.022581,0.030935,0.031323,0.174,0.177419,38.0,0.298742,0.012194,-13.1,11.7
5996,7118,627817,6442868,1960-01-01,"1,1",POINT (579999.298 2008998.399),22.7,119.8,2.783871,1.996774,...,4.622581,0.032935,0.033581,0.180,0.203226,50.2,0.264548,0.012129,-12.8,12.2
5995,7117,619825,6442935,1960-01-01,"1,095",POINT (571999.291 2008998.458),21.8,115.4,3.277419,2.083871,...,4.332258,0.030839,0.031226,0.170,0.180645,45.7,0.284097,0.010065,-12.2,13.0
5994,7116,611833,6443002,1960-01-01,"1,086",POINT (563999.284 2008998.565),23.8,108.4,3.490323,2.174194,...,4.690323,0.032871,0.033323,0.179,0.183871,41.7,0.293774,0.008742,-11.5,13.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6999211,3831,805955,6713138,2024-12-01,"0,927",POINT (756000.273 2280998.1),0.9,68.4,4.016129,2.809677,...,0.000000,0.000000,0.000000,0.000,0.000000,0.9,0.344613,0.000097,-3.9,12.8
6999212,3832,813948,6713070,2024-12-01,"0,908",POINT (763999.644 2280998.186),1.5,71.1,2.861290,3.348387,...,0.000000,0.000000,0.000000,0.001,0.000000,1.8,0.324968,0.000613,-5.0,11.4
6999213,3833,821942,6713002,2024-12-01,"0,923",POINT (772000.016 2280998.33),3.2,71.2,2.835484,3.409677,...,0.000000,0.000000,0.000000,0.001,0.000000,3.9,0.321000,0.000516,-4.8,11.5
6999207,3827,773980,6713410,2024-12-01,"0,973",POINT (723999.802 2280998.23),0.0,64.9,4.154839,2.825806,...,0.000000,0.000000,0.000000,0.000,0.000000,0.0,0.375032,0.000774,-2.5,13.3


In [ ]:
print("CRS gdf_join :", gdf_join.crs)
print("CRS argile   :", argile.crs)

if argile.crs != gdf_join.crs:
    argile = argile.to_crs(gdf_join.crs)
    

cols_argile = ["geometry", "ALEA", "NIVEAU", "DPT"]


CRS gdf_join : EPSG:27572
CRS argile   : EPSG:27572


In [ ]:
output_fp = Path("data") / "processed" / "jointure_meteo_swi_argile_nearest.gpkg"
output_fp.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Reproject swi_monthly to match the CRS of argile
print(swi_monthly.crs)
cols_argile = ["geometry", "ALEA", "NIVEAU", "DPT"]
argile=argile.to_crs(swi_monthly.crs)

swi_monthly_argile = gpd.sjoin(
    swi_monthly,
    argile[cols_argile],
    how="left",
    predicate="intersects"  # ou "within", l’un ou l’autre pour des points dans des polygones
)


EPSG:27572


In [ ]:
swi_monthly_argile

,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry,index_right,ALEA,NIVEAU,DPT
0,1958-08-01,600,24010,0.0,101.1,16.183871,3.735484,10.297871,104077.3,31484.2,...,0.0,0.234903,0.0,10.3,21.4,POINT (60000 2401000),NaN,NaN,NaN,NaN
1,1958-08-01,760,23610,0.0,85.0,16.277419,4.748387,10.441677,103165.5,31824.6,...,0.0,0.174645,0.0,11.5,21.5,POINT (76000 2361000),NaN,NaN,NaN,NaN
2,1958-08-01,760,23930,0.0,84.1,16.400000,4.719355,10.511032,103705.2,31824.6,...,0.0,0.226355,0.0,11.6,21.8,POINT (76000 2393000),NaN,NaN,NaN,NaN
3,1958-08-01,760,24010,0.0,103.3,15.819355,3.800000,10.094516,102541.1,31493.2,...,0.0,0.220677,0.0,10.2,20.6,POINT (76000 2401000),NaN,NaN,NaN,NaN
4,1958-08-01,760,24090,0.0,102.4,15.958065,3.764516,10.168677,103106.1,31489.5,...,0.0,0.217355,0.0,10.2,20.9,POINT (76000 2409000),NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7982859,2025-10-01,11960,17050,0.0,44.5,18.122581,2.538710,8.597774,91740.2,33351.6,...,0.0,0.190774,0.0,12.5,25.8,POINT (1196000 1705000),1180349.0,Faible,1.0,202.0
7982860,2025-10-01,11960,17130,0.0,27.8,16.987097,2.683871,7.945290,92372.0,33203.6,...,0.0,0.194065,0.0,9.3,25.8,POINT (1196000 1713000),1180449.0,Moyen,2.0,202.0
7982861,2025-10-01,11960,17210,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.0,0.186806,0.0,9.3,25.8,POINT (1196000 1721000),1180520.0,Faible,1.0,202.0
7982862,2025-10-01,11960,17290,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.0,0.204161,0.0,9.3,25.8,POINT (1196000 1729000),NaN,NaN,NaN,NaN


In [10]:

# Perform a spatial join using the nearest method
swi_monthly_argile_nearest = gpd.sjoin_nearest(
    swi_monthly,
    argile[cols_argile],
    how="left",
    distance_col="dist_to_argile"
)



In [ ]:
swi_monthly_argile_nearest

,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry,index_right,ALEA,NIVEAU,DPT,dist_to_argile
0,1958-08-01,600,24010,0.0,101.1,16.183871,3.735484,10.297871,104077.3,31484.2,...,0.234903,0.0,10.3,21.4,POINT (60000 2401000),27,Faible,1.0,29,2063.770829
1,1958-08-01,760,23610,0.0,85.0,16.277419,4.748387,10.441677,103165.5,31824.6,...,0.174645,0.0,11.5,21.5,POINT (76000 2361000),331,Faible,1.0,29,683.329171
2,1958-08-01,760,23930,0.0,84.1,16.400000,4.719355,10.511032,103705.2,31824.6,...,0.226355,0.0,11.6,21.8,POINT (76000 2393000),156,Faible,1.0,29,1021.905694
3,1958-08-01,760,24010,0.0,103.3,15.819355,3.800000,10.094516,102541.1,31493.2,...,0.220677,0.0,10.2,20.6,POINT (76000 2401000),168,Faible,1.0,29,347.199910
4,1958-08-01,760,24090,0.0,102.4,15.958065,3.764516,10.168677,103106.1,31489.5,...,0.217355,0.0,10.2,20.9,POINT (76000 2409000),135,Faible,1.0,29,232.342152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7982859,2025-10-01,11960,17050,0.0,44.5,18.122581,2.538710,8.597774,91740.2,33351.6,...,0.190774,0.0,12.5,25.8,POINT (1196000 1705000),1180349,Faible,1.0,202,0.000000
7982860,2025-10-01,11960,17130,0.0,27.8,16.987097,2.683871,7.945290,92372.0,33203.6,...,0.194065,0.0,9.3,25.8,POINT (1196000 1713000),1180449,Moyen,2.0,202,0.000000
7982861,2025-10-01,11960,17210,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.186806,0.0,9.3,25.8,POINT (1196000 1721000),1180520,Faible,1.0,202,0.000000
7982862,2025-10-01,11960,17290,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.204161,0.0,9.3,25.8,POINT (1196000 1729000),1180530,Faible,1.0,202,620.374701


In [26]:
meteo_france

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,geometry
0,2,641374,7106309,1960-01-01,"0,863",POINT (641374 7106309)
1,2,641374,7106309,1960-02-01,"0,876",POINT (641374 7106309)
2,2,641374,7106309,1960-03-01,"0,856",POINT (641374 7106309)
3,2,641374,7106309,1960-04-01,"0,757",POINT (641374 7106309)
4,2,641374,7106309,1960-05-01,"0,673",POINT (641374 7106309)
...,...,...,...,...,...,...
7005175,9892,1215772,6046242,2024-08-01,"-0,019",POINT (1215772 6046242)
7005176,9892,1215772,6046242,2024-09-01,"0,007",POINT (1215772 6046242)
7005177,9892,1215772,6046242,2024-10-01,"0,17",POINT (1215772 6046242)
7005178,9892,1215772,6046242,2024-11-01,"0,126",POINT (1215772 6046242)
